In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\projects\\project-9 kidney disease\\kidney-desease-classification_again\\kidney-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    param_image_size: list
    param_learning_rate: float
    param_include_top: bool
    param_weights: str
    param_model_name: str
    params_classes: int

In [ ]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath =  CONFIG_FILE_PATH,
        params_filepath =  PARAMS_FILE_PATH
        ):
                     
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        


    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir = Path(config.root_dir),
            base_model_path = Path(config.base_model_path),
            updated_base_model_path = Path(config.updated_base_model_path),
            param_image_size = self.params.IMAGE_SIZE,
            param_learning_rate = self.params.LEARNING_RATE,
            param_include_top = self.params.INCLUDE_TOP,
            param_weights = self.params.WEIGHTS,
            param_model_name = self.params.MODEL_NAME,
            params_classes = self.params.CLASSES
            
        )
        
        return prepare_base_model_config

In [ ]:
import os
import urllib.request as request
from zipfile import ZipFile
from src.cnnClassifier import logger
import tensorflow as tf


In [ ]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        logger.info("Downloading base model")
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape = self.config.param_image_size,
            include_top = self.config.param_include_top,
            weights = self.config.param_weights
        )

        logger.info("Saving base model")
        base_model = self.model
        
        self.save_model(path=self.config.base_model_path, model=base_model)

        
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable= False
        elif(freeze_till is not None) and (freeze_till> 0):
            for layer in model.layers[:-freeze_till]:
                 model.trainable = False
                 
        flatten_in = tf.keras.layers.Flatten()(model.output)
        x = tf.keras.layers.Dense(256, activation="relu")(flatten_in)
        x = tf.keras.layers.Dropout(0.5)(x)
        output = tf.keras.layers.Dense(units=1, activation="sigmoid")(x)
        
        full_model = tf.keras.models.Model(
            inputs = model.input,
            outputs = output
        )
        
        full_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.BinaryCrossentropy(),
            metrics = ["accuracy"]
        )
        
        full_model.summary()
        return full_model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
        model = self.model,
        classes= self.config.params_classes,
        freeze_all = False,
        freeze_till = 4,
        learning_rate = self.config.param_learning_rate
        )
        
        self.save_model(path= self.config.updated_base_model_path, model = self.full_model)
        
    @staticmethod
    
    def save_model(path:Path, model: tf.keras.Model):
        model.save(path)
    

In [9]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config = prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-04-04 11:36:21,093: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-04 11:36:21,116: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-04 11:36:21,127: INFO: common: created directory at : artifacts]
[2026-04-04 11:36:21,130: INFO: common: created directory at : artifacts/prepare_base_model]
[2026-04-04 11:36:21,135: INFO: 1109691473: Downloading base model]
[2026-04-04 11:36:24,378: INFO: 1109691473: Saving base model]
[2026-04-04 11:36:24,384: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 6